In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/EduHinglish-ollama

/content/drive/MyDrive/EduHinglish-ollama


In [3]:
!ls
!ls training
!ls data

data  training
enrich_hinglish.py
ss9.json  ss9.json.broken_backup


In [4]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 69 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (589 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [38]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

!ollama list

NAME           ID              SIZE      MODIFIED          
qwen2.5:14b    7cdf5a0187d5    9.0 GB    56 minutes ago       
qwen3:8b       500a1f067a9f    5.2 GB    About an hour ago    


In [7]:
!nvidia-smi

Fri Sep 11 03:00:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
!ollama pull qwen2.5:14b

In [8]:
!ollama pull qwen3:8b

In [ ]:
!ollama run qwen3:8b "Explain the water cycle in simple Hinglish for a Class 9 student."

In [ ]:
!ollama run qwen3:8b "Explain photosynthesis in simple Hinglish in two sentences."

In [9]:
!ollama run qwen3:8b "hi" &
import time; time.sleep(3)
!ollama ps

Thinking...
Okay, the user said "hi /think". I need to respond appropriately. First, I 
should acknowledge their greeting. Since they included "/think", maybe they
they're testing if I can process that command. I should check if there's a 
specific action related to "/think" that I need to take. If not, just respo
respond politely. Let me make sure there's no hidden instruction here. The 
main goal is to greet them back and offer assistance. Keep it friendly and 
open-ended.
...done thinking.

Hello! How can I assist you today? 😊

NAME        ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3:8b    500a1f067a9f    5.6 GB    100% GPU     4096       4 minutes from now    


In [11]:
# check for broken lines in json if any
path = "data/ss9.json"
with open(path, "r", encoding="utf-8") as f:
    raw = f.read()

import json
try:
    data = json.loads(raw)
    print(f"Valid JSON, {len(data)} entries — false alarm, re-run the script.")
except json.JSONDecodeError as e:
    print(f"Broken at line {e.lineno}, col {e.colno}, char {e.pos}")
    print(f"File size: {len(raw)} chars")
    print("--- 300 chars before/after the break ---")
    print(raw[max(0, e.pos-300):e.pos+300])

Valid JSON, 3668 entries — false alarm, re-run the script.


In [ ]:
# fix those broken lines
import json, shutil

path = "data/ss9.json"
with open(path, "r", encoding="utf-8") as f:
    raw = f.read()

depth = 0
in_string = False
escape = False
last_safe_end = None  # position right after the last fully-closed top-level entry

for i, ch in enumerate(raw):
    if in_string:
        if escape:
            escape = False
        elif ch == '\\':
            escape = True
        elif ch == '"':
            in_string = False
        continue
    if ch == '"':
        in_string = True
    elif ch in '{[':
        depth += 1
    elif ch in '}]':
        depth -= 1
        if depth == 1 and ch == '}':
            last_safe_end = i + 1  # closed a top-level entry (array itself is depth 1)

if last_safe_end is None:
    print("No complete entry found — something else is wrong, don't proceed.")
else:
    recovered_text = raw[:last_safe_end] + "\n]"
    data = json.loads(recovered_text)  # will raise if still broken — that's fine, tells us more work is needed
    print(f"Recovered {len(data)} valid entries out of the intended set "
          f"(kept {last_safe_end} of {len(raw)} chars — lost only the tail that was mid-write).")

    shutil.copy(path, path + ".broken_backup")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("Saved recovered file. Original broken copy kept at:", path + ".broken_backup")

In [28]:
# clear alll hinglish question & answers
import json

path = "data/ss9.json"
with open(path, encoding="utf-8") as f:
    data = json.load(f)

for e in data:
    e["question_hinglish"] = ""
    e["answer_hinglish"] = ""
    e["messages_hinglish"] = []

with open(path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Cleared question_hinglish / answer_hinglish / messages_hinglish for all {len(data)} entries")

Cleared question_hinglish / answer_hinglish / messages_hinglish for all 3668 entries


In [ ]:
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen3:8b --concurrency 1 --num-ctx 3072 --num-predict 1500 --limit 20

In [ ]:
# qwen2.5:7b 20 entries
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen2.5:7b --concurrency 1 --num-ctx 3072 --num-predict 1500 --limit 20

In [31]:
# qwen2.5:14b 20 entries
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen2.5:14b --concurrency 1 --num-ctx 3072 --num-predict 1500 --limit 20


[EduHinglish Enrichment] -> ss9.json
  Backend    : ollama
  Model      : qwen2.5:14b
  Concurrency: 1
  Checkpoint : every 20 entries

  Reuse cache seeded: 0 unique questions, 0 unique answers
  Total entries : 3668
  Already filled: 0
  To process    : 20 (limited to 20)

  [20/20] checkpoint saved  |  0.2 entries/s  |  ~0.0 min remaining

Done! Enriched 20 entries -> /content/drive/MyDrive/EduHinglish-ollama/data/ss9.json


In [ ]:
# qwen3:8b 1 entry
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen3:8b --concurrency 1 --limit 1

In [ ]:
# qwen3:8b 20 entries
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen3:8b --concurrency 20 --limit 20

In [ ]:
# qwen3:8b 20 entries
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen3:8b --concurrency 2 --save-every 20

In [39]:
# qwen2.5:14b 20 entries
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen2.5:14b --concurrency 2 --save-every 20


[EduHinglish Enrichment] -> ss9.json
  Backend    : ollama
  Model      : qwen2.5:14b
  Concurrency: 2
  Checkpoint : every 20 entries

  Reuse cache seeded: 20 unique questions, 7 unique answers
  Total entries : 3668
  Already filled: 20
  To process    : 3648

  [20/3648] checkpoint saved  |  0.2 entries/s  |  ~322.8 min remaining
  [40/3648] checkpoint saved  |  0.2 entries/s  |  ~268.8 min remaining
  [60/3648] checkpoint saved  |  0.2 entries/s  |  ~286.9 min remaining
Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/base_events.py", line 726, in run_until_complete
    return future.result()
           ~~~~~~~~~~~~~^^
  File "/content/drive/MyDrive/EduHinglish-ollama/training/enrich_hinglish.py", line 402, in run
    results = await asyncio.gather(*tasks)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  

In [ ]:
# qwen3:8b 50 entries
!python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen3:8b --concurrency 1 --limit 50

In [ ]:
# # qwen3:8b 20 entries for continuous running
!bash -c 'while true; do ollama serve >/tmp/ollama.log 2>&1 & OLLAMA_PID=$!; sleep 5; while kill -0 $OLLAMA_PID 2>/dev/null; do sleep 5; done; done & python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen3:8b --concurrency 2 --save-every 20'


In [ ]:
# # qwen2.5:14b 20 entries for continuous running
!bash -c 'while true; do ollama serve >/tmp/ollama.log 2>&1 & OLLAMA_PID=$!; sleep 5; while kill -0 $OLLAMA_PID 2>/dev/null; do sleep 5; done; done & python training/enrich_hinglish.py --input data/ss9.json --backend ollama --model qwen2.5:14b --concurrency 2 --save-every 20'



[EduHinglish Enrichment] -> ss9.json
  Backend    : ollama
  Model      : qwen2.5:14b
  Concurrency: 2
  Checkpoint : every 20 entries

  Reuse cache seeded: 80 unique questions, 21 unique answers
  Total entries : 3668
  Already filled: 80
  To process    : 3588

  [20/3588] checkpoint saved  |  0.1 entries/s  |  ~431.5 min remaining
  [40/3588] checkpoint saved  |  0.2 entries/s  |  ~315.8 min remaining
  [60/3588] checkpoint saved  |  0.2 entries/s  |  ~288.4 min remaining
  [80/3588] checkpoint saved  |  0.2 entries/s  |  ~264.8 min remaining
  [100/3588] checkpoint saved  |  0.2 entries/s  |  ~252.4 min remaining
  [120/3588] checkpoint saved  |  0.2 entries/s  |  ~262.0 min remaining
  [140/3588] checkpoint saved  |  0.2 entries/s  |  ~260.4 min remaining
  [160/3588] checkpoint saved  |  0.2 entries/s  |  ~264.5 min remaining
  [180/3588] checkpoint saved  |  0.2 entries/s  |  ~305.2 min remaining
  [200/3588] checkpoint saved  |  0.2 entries/s  |  ~312.4 min remaining
  [220/3

In [ ]:
!grep -n "STATEMENT_PROMPT\|split_true_false" training/enrich_hinglish.py

In [ ]:
# print only true false questions for quick evaluation
import json
with open("data/ss9.json", encoding="utf-8") as f:
    data = json.load(f)
for e in data:
    if e["question_english"].strip().lower().startswith("true or false"):
        print(e["question_english"])
        print("->", e["question_hinglish"])
        print()

In [33]:
# get first 20 entries
import json

with open("data/ss9.json", encoding="utf-8") as f:
    data = json.load(f)

for i, e in enumerate(data[:20]):
    print(f"--- {i+1} ---")
    print("EN Q:", e["question_english"])
    print("HI Q:", e["question_hinglish"])
    print("EN A:", e["answer_english"])
    print("HI A:", e["answer_hinglish"])
    print()

--- 1 ---
EN Q: True or False: In Grades 6 to 8, we have explored Social Science through stories of people, places, and events.
HI Q: 6th se 8th tak humne logon, jagahon aur incidenton ke kahaniyon ke through Social Science explore kiya hai. — Sahi ya galat?
EN A: True. In Grades 6 to 8, we have explored Social Science through stories of people, places, and events as stated in the NCERT textbook.
HI A: 6 to 8 class mein hum NCERT book ke through logon, jagahon aur incidenton ke stories se Social Science ko explore kiya hai, yaad rakhna.

--- 2 ---
EN Q: True or False: As you enter Grade 9, it is time to pause and understand what ‘Social Science’ truly means and why it matters.
HI Q: Grade 9 mein aate hue pause karo aur socho ki 'Social Science' kya hai aur yeh kyun important hai. — Sahi ya galat?
EN A: True. As you enter Grade 9, it is time to pause and understand what ‘Social Science’ truly means and why it matters as stated in the NCERT textbook.
HI A: Ab class 9 mein aake socho, kya

In [36]:
import json, collections

with open("data/ss9.json", encoding="utf-8") as f:
    data = json.load(f)

counts = collections.Counter(e["answer_hinglish"] for e in data if e.get("answer_hinglish"))
print(f"{len(data)} total entries -> {len(counts)} unique answer translations\n")

for text, n in counts.most_common(30):
    print(f"[{n} entries] {text[:120]}...")

3668 total entries -> 7 unique answer translations

[7 entries] 6th se 8th tak humne Social Science ke logon, jagahon aur incidenton ke stories ke through explore kiye the. Ab 9th mein...
[7 entries] Jeevan humse pehle hota hai woh environment, institutions jo hum ko govern karte hain, economic activities jo humare nee...
[2 entries] Yeh sirf kya hua ya kahan hai ke baad, kyun hua, log kaise jeete hain, government kaise kaam karta hai, ekonomi kaise ch...
[1 entries] 6 to 8 class mein hum NCERT book ke through logon, jagahon aur incidenton ke stories se Social Science ko explore kiya h...
[1 entries] Ab class 9 mein aake socho, kya hai ‘Social Science’ aur kyun important hai, jo NCERT book mein bataya gaya hai....
[1 entries] Yahin se pata chalta hai ki Social Science kya hai. Isme hum dekhte hain ki yeh connections kitne complex hain aur unka ...
[1 entries] Social Science toh basically human society ke baare mein systematic study hai, yeh NCERT book mein kahaa gaya hai....
